# Week 23 · Notebook 3: AI Functions & Genie Agents

# Requirements: Databricks workspace (free trial) + upload the week-01 CSVs to a volume

Upload into `/Volumes/zrl_/zorologistics/raw/`:

- `support_tickets.csv`

The AI-function cells below are **SERVERLESS ONLY**: AI functions require serverless compute and Databricks Runtime **18.2+** (they will not run on Pro/Classic warehouses). Run them on a serverless SQL warehouse.


## AI functions, in SQL

**AI functions** put LLMs directly into SQL with no endpoint or API key to manage. This notebook demos `ai_classify`, `ai_extract`, `ai_mask`, `ai_query`, and `ai_analyze_sentiment`, then walks through standing up a **Genie Agents** space. Requirements (Aug 2026): serverless compute + DBR 18.2+. See research §4.3 and `reference/platforms/databricks/12-ai-functions-genie.md`.


In [ ]:
%sql
-- Load the support tickets for classification demos.
CREATE OR REPLACE TABLE zrl_.zorologistics.support_tickets AS
SELECT *
FROM read_files('/Volumes/zrl_/zorologistics/raw/support_tickets.csv',
                format => 'csv', header => true, inferSchema => true)


## `ai_classify`: ticket category from free text

`ai_classify(content, labels)` returns the best-matching label. We use it to route a ticket to the right team.


In [ ]:
%sql
-- SERVERLESS ONLY (serverless + DBR 18.2+).
-- ai_classify: assign each ticket one of the provided category labels.
SELECT
  ticket_id,
  ai_classify(text, ARRAY('damage', 'refund', 'tracking', 'billing', 'customs', 'documents')) AS category_ai
FROM zrl_.zorologistics.support_tickets
LIMIT 10


## `ai_analyze_sentiment`: positive/negative/neutral/mixed

A quick triage signal for the support queue.


In [ ]:
%sql
-- SERVERLESS ONLY.
SELECT ticket_id, ai_analyze_sentiment(text) AS sentiment
FROM zrl_.zorologistics.support_tickets
LIMIT 10


## `ai_extract`: structured fields from a bill of lading

`ai_extract(content, schema)` pulls structured fields from unstructured text. We stage two BoL samples and extract typed fields.


In [ ]:
from pyspark.sql import Row

bols = [
  ("ZRL-10000", "BILL OF LADING No. ZRL-10000\nSHIPPER: Atlas Freight Logistics Div.\n"
    "CONSIGNEE: ZoroLogistics Customer #000\nPORT OF LOADING: Long Beach\n"
    "PORT OF DISCHARGE: Miami\nCOMMODITY: electronics\nQUANTITY: 3 pallets\n"
    "GROSS WEIGHT: 1152 KG\nDECLARED VALUE: USD 23410.50\nFREIGHT TERMS: PREPAID"),
  ("ZRL-10001", "BILL OF LADING No. ZRL-10001\nSHIPPER: BlueHarbor Lines Logistics Div.\n"
    "CONSIGNEE: ZoroLogistics Customer #001\nPORT OF LOADING: Oakland\n"
    "PORT OF DISCHARGE: Savannah\nCOMMODITY: auto parts\nQUANTITY: 12 pallets\n"
    "GROSS WEIGHT: 4200 KG\nDECLARED VALUE: USD 91000.00\nFREIGHT TERMS: COLLECT"),
]
spark.createDataFrame([Row(bol_id=i, text=t) for i, t in bols]).createOrReplaceTempView("bol_samples")
print("BoL samples staged:", len(bols))


In [ ]:
%sql
-- SERVERLESS ONLY.
-- ai_extract: pull structured fields from the BoL text using a schema you define.
SELECT
  bol_id,
  ai_extract(text,
    'shipper STRING, port_of_loading STRING, commodity STRING, quantity INT, '
    'gross_weight_kg DOUBLE, declared_value_usd DOUBLE, freight_terms STRING') AS fields
FROM bol_samples


## `ai_mask`: redact PII

`ai_mask(content, labels)` masks the entity types you name (e.g. person names, emails). Use it before support text leaves the warehouse.


In [ ]:
%sql
-- SERVERLESS ONLY.
-- ai_mask: redact the named PII entity types from free text.
SELECT ai_mask(
  'Customer Jane Doe reached out from jane.doe@example.com about shipment S0000042',
  ARRAY('person_name', 'email')
) AS masked


## `ai_query`: any prompt against a foundation model

`ai_query(endpoint, prompt)` is the general-purpose escape hatch: route any prompt to a supported Foundation Model API endpoint.


In [ ]:
%sql
-- SERVERLESS ONLY.
SELECT ai_query(
  'databricks-meta-llama-3-3-70b-instruct',
  'Summarize the ZoroLogistics refund policy in one sentence.'
) AS summary


## Genie Agents: setup checklist

Genie Agents (formerly Genie Spaces) turn curated tables into a natural-language Q&A space that business analysts can query. Creation is a UI flow; the discipline is the **verified answers** you seed so its SQL is trustworthy.

**Setup steps**

1. **Create a space**: New → Genie → "ZoroLogistics Ops". Pick the datasets (`gold_on_time_kpis`, `silver_shipments`, `carriers`, `lanes`).
2. **Write instructions**: e.g.: *"on_time_rate is the fraction of shipments arriving within 2 hours of plan. month is a yyyy-MM string. delay is measured in hours."*
3. **Add SQL functions**: expose UDFs (e.g. `zrl_.zorologistics.mask_value`) so Genie can reuse governed logic.
4. **Seed example questions**: "Which lane had the lowest on-time rate last month?" with the exact SQL.
5. **Add verified answers**: for each example, mark the returned SQL as *verified* after you confirm it.
6. **Publish**: share the space with the ops team and let them ask in plain English.

**Verified-answer checklist**

- [ ] The generated SQL returns the same numbers as the gold table query.
- [ ] Aggregations use `avg(is_on_time)` semantics, not raw counts.
- [ ] No PII column (`customer_id`, `text`) is reachable without a mask.
- [ ] "month" is treated as a string, not a date, in filters.


In [ ]:
# Final metric: ticket volume + the high/critical fraction we would route with ai_classify.
n = spark.sql("SELECT count(*) FROM zrl_.zorologistics.support_tickets").collect()[0][0]
pct = spark.sql(
    "SELECT round(avg(CASE WHEN priority IN ('high','critical') THEN 1.0 ELSE 0.0 END), 4) "
    "FROM zrl_.zorologistics.support_tickets").collect()[0][0]
print("support tickets:", n)
print("high/critical fraction:", pct)
